In [ ]:

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None — go to Runtime > Change runtime type > T4 GPU")

!pip install transformers datasets scikit-learn pandas numpy matplotlib seaborn -q

import transformers
print("Transformers version:", transformers.__version__)

In [ ]:
!pip install kaggle -q
from google.colab import files
files.upload()  # upload your kaggle.json API key here

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d jp797498e/twitter-entity-sentiment-analysis --unzip

In [ ]:

import pandas as pd
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader
import torch

# Load dataset
# Upload the Kaggle CSV first (twitter_training.csv / twitter_validation.csv)

train_df = pd.read_csv("twitter_training.csv", header=None,
                        names=["id", "entity", "sentiment", "text"])
val_df   = pd.read_csv("twitter_validation.csv", header=None,
                        names=["id", "entity", "sentiment", "text"])

label_map = {"Positive": 2, "Neutral": 1, "Negative": 0, "Irrelevant": 3}
for df in [train_df, val_df]:
    df.dropna(subset=["text", "sentiment"], inplace=True)
    df["label"] = df["sentiment"].map(label_map)

print(train_df["sentiment"].value_counts())

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(texts, max_len=64):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

class TweetDataset(Dataset):
    def __init__(self, df):
        encoded = tokenize(df["text"])
        self.input_ids      = encoded["input_ids"]
        self.attention_mask = encoded["attention_mask"]
        self.labels         = torch.tensor(df["label"].values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels":         self.labels[idx],
        }

train_dataset = TweetDataset(train_df)
val_dataset   = TweetDataset(val_df)

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)}")

sample = train_dataset[0]
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

In [ ]:
# All 74K samples, 3 epochs, proper hyperparameters

import torch, gc, time
from transformers import DistilBertForSequenceClassification, get_scheduler
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_LABELS  = 4
EPOCHS      = 3
BATCH_SIZE  = 64              # bump up since we have GPU memory
LR          = 2e-5

print(f"Device: {DEVICE}")
assert DEVICE.type == "cuda", "STOP — GPU not active. Fix runtime before continuing."

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=NUM_LABELS
).to(DEVICE)

print(f"Model on: {next(model.parameters()).device}")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
num_steps = len(train_loader) * EPOCHS
scheduler = get_scheduler("linear", optimizer=optimizer,
                           num_warmup_steps=num_steps // 10,
                           num_training_steps=num_steps)

# ── Training loop
history = {"train_loss": [], "val_acc": [], "val_f1_w": [], "val_f1_m": []}
class_names = ["Negative", "Neutral", "Positive", "Irrelevant"]

total_start = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()
    model.train()
    total_loss = 0

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        outputs = model(**batch)
        outputs.loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += outputs.loss.item()

        # Progress print every 100 batches
        if (step + 1) % 100 == 0:
            print(f"  Epoch {epoch+1} | batch {step+1}/{len(train_loader)} | running loss: {total_loss/(step+1):.4f}")

    avg_loss = total_loss / len(train_loader)

    # ── Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            logits = model(**batch).logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch["labels"].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1_w = f1_score(all_labels, all_preds, average="weighted")
    f1_m = f1_score(all_labels, all_preds, average="macro")

    history["train_loss"].append(avg_loss)
    history["val_acc"].append(acc)
    history["val_f1_w"].append(f1_w)
    history["val_f1_m"].append(f1_m)

    epoch_time = time.time() - epoch_start
    print(f"\nEpoch {epoch+1}/{EPOCHS} ({epoch_time:.1f}s) | "
          f"Loss: {avg_loss:.4f} | Acc: {acc:.4f} | Weighted F1: {f1_w:.4f} | Macro F1: {f1_m:.4f}\n")

total_time = time.time() - total_start
print(f"=== Total training time: {total_time/60:.1f} min ===")

# Prediction counts per class
pred_counts = np.bincount(all_preds, minlength=4)
print("\nPrediction counts per class:")
for name, count in zip(class_names, pred_counts):
    print(f"  {name}: {count}")

# ── Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("DistilBERT (Full Training) — Confusion Matrix")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
#Classical ML Baselines (TF-IDF + SVM and Logistic Regression)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time

# ── TF-IDF features
print("Building TF-IDF features...")
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),       # unigrams + bigrams
    min_df=2,                  # ignore super-rare terms
    sublinear_tf=True,
)

X_train = tfidf.fit_transform(train_df["text"])
X_val   = tfidf.transform(val_df["text"])
y_train = train_df["label"].values
y_val   = val_df["label"].values

print(f"TF-IDF matrix shape: {X_train.shape}")

class_names = ["Negative", "Neutral", "Positive", "Irrelevant"]
results = {}

# ── Model 1: Linear SVM
print("\n" + "="*50)
print("Training TF-IDF + Linear SVM...")
print("="*50)
t0 = time.time()
svm = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm.fit(X_train, y_train)
svm_time = time.time() - t0

svm_preds = svm.predict(X_val)
svm_acc  = accuracy_score(y_val, svm_preds)
svm_f1_w = f1_score(y_val, svm_preds, average="weighted")
svm_f1_m = f1_score(y_val, svm_preds, average="macro")

print(f"Train time: {svm_time:.1f}s")
print(f"Accuracy: {svm_acc:.4f} | Weighted F1: {svm_f1_w:.4f} | Macro F1: {svm_f1_m:.4f}")
results["SVM"] = {"acc": svm_acc, "f1_w": svm_f1_w, "f1_m": svm_f1_m, "time": svm_time, "preds": svm_preds}

# ── Model 2: Logistic Regression
print("\n" + "="*50)
print("Training TF-IDF + Logistic Regression...")
print("="*50)
t0 = time.time()
lr = LogisticRegression(max_iter=2000, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)
lr_time = time.time() - t0

lr_preds = lr.predict(X_val)
lr_acc  = accuracy_score(y_val, lr_preds)
lr_f1_w = f1_score(y_val, lr_preds, average="weighted")
lr_f1_m = f1_score(y_val, lr_preds, average="macro")

print(f"Train time: {lr_time:.1f}s")
print(f"Accuracy: {lr_acc:.4f} | Weighted F1: {lr_f1_w:.4f} | Macro F1: {lr_f1_m:.4f}")
results["LogReg"] = {"acc": lr_acc, "f1_w": lr_f1_w, "f1_m": lr_f1_m, "time": lr_time, "preds": lr_preds}

# ── Summary
print("\n" + "="*50)
print("SUMMARY — Classical Baselines")
print("="*50)
print(f"{'Model':<25} {'Accuracy':<12} {'Weighted F1':<14} {'Macro F1':<12} {'Time':<8}")
print("-"*75)
print(f"{'TF-IDF + SVM':<25} {svm_acc:<12.4f} {svm_f1_w:<14.4f} {svm_f1_m:<12.4f} {svm_time:<.1f}s")
print(f"{'TF-IDF + LogReg':<25} {lr_acc:<12.4f} {lr_f1_w:<14.4f} {lr_f1_m:<12.4f} {lr_time:<.1f}s")

# ── Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, info) in zip(axes, [("SVM", results["SVM"]), ("Logistic Regression", results["LogReg"])]):
    cm = confusion_matrix(y_val, info["preds"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"TF-IDF + {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.show()

# ── Per-class breakdown
print("\n\nClassification Report — SVM:")
print(classification_report(y_val, svm_preds, target_names=class_names))

print("\nClassification Report — Logistic Regression:")
print(classification_report(y_val, lr_preds, target_names=class_names))